# Phase 4: Portfolio Construction & Risk Management  
## Notebook 04_09 — Risk Parity and Equal Risk Contribution

### Research question

How do we construct portfolios whose risk is diversified across assets or asset classes, rather than concentrated in the highest-volatility positions?

This notebook follows:

```text
04_03_volatility_targeting_and_position_sizing
04_04_mean_variance_optimization_ledoit_wolf
04_05_pca_spectral_risk_analysis
04_06_var_cvar_and_stress_testing
04_08_futures_minimum_variance_hedge_ratio
```

The previous notebooks showed volatility targeting, covariance estimation, PCA risk modes, tail risk, and hedging. This notebook asks:

> Can we allocate without making fragile expected-return forecasts, by focusing instead on risk contributions?

It covers:

1. equal weight versus equal risk;
2. portfolio volatility;
3. marginal risk contribution;
4. component risk contribution;
5. equal risk contribution objective;
6. inverse-volatility allocation;
7. full-covariance risk parity;
8. asset-class risk budgets;
9. long-only ERC optimisation;
10. covariance shrinkage fallback;
11. rolling risk parity;
12. turnover and transaction costs;
13. realised risk contribution;
14. stress-regime behaviour;
15. leverage and volatility targeting;
16. comparison with mean-variance and GMV;
17. diversification ratio;
18. audit checks;
19. saved outputs and manifest.

The key idea:

> Equal capital weights are not equal risk weights. Risk parity allocates capital so that each sleeve contributes a controlled share of portfolio risk.

## 1. Portfolio volatility

For weights $w$ and covariance matrix $\Sigma$:

$$
\sigma_p = \sqrt{w^\top \Sigma w}
$$

This is the total portfolio volatility.

But knowing total volatility is not enough.

We also want to know:

> Which assets are responsible for that volatility?

## 2. Marginal and component risk contribution

Portfolio volatility:

$$
\sigma_p = \sqrt{w^\top \Sigma w}
$$

Marginal risk contribution:

$$
MRC_i = \frac{\partial \sigma_p}{\partial w_i} = \frac{(\Sigma w)_i}{\sigma_p}
$$

Component risk contribution:

$$
RC_i = w_i \cdot MRC_i = w_i \frac{(\Sigma w)_i}{\sigma_p}
$$

Risk contribution as a fraction of total risk:

$$
\%RC_i = \frac{RC_i}{\sigma_p}
$$

The contributions sum to one:

$$
\sum_i \%RC_i = 1
$$

## 3. Equal risk contribution

Equal Risk Contribution, or ERC, targets:

$$
\%RC_i = \frac{1}{N}
$$

for all assets $i$.

More generally, for risk budgets $b_i$:

$$
\%RC_i = b_i
$$

where:

$$
\sum_i b_i = 1
$$

This is useful when a portfolio wants, for example:

- 30% risk from equities;
- 25% from bonds;
- 25% from commodities;
- 10% from trend;
- 10% from crypto or alternatives.

Risk parity is not return forecasting. It is risk allocation.

## 4. Risk parity versus inverse volatility

If assets are uncorrelated, inverse-volatility weights approximate equal risk contribution:

$$
w_i \propto \frac{1}{\sigma_i}
$$

But if correlations are meaningful, inverse-volatility ignores the covariance structure.

Full-covariance ERC uses:

$$
\Sigma
$$

and therefore accounts for correlation.

This notebook compares:

1. equal weight;
2. inverse volatility;
3. global minimum variance;
4. equal risk contribution;
5. asset-class risk-budget ERC.

## 5. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from scipy.optimize import minimize
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False

try:
    from sklearn.covariance import LedoitWolf
    SKLEARN_AVAILABLE = True
except Exception:
    SKLEARN_AVAILABLE = False

SCIPY_AVAILABLE, SKLEARN_AVAILABLE

In [ ]:
@dataclass(frozen=True)
class RiskParityConfig:
    n_days: int = 2_400
    annualisation: int = 252
    train_fraction: float = 0.65
    estimation_window: int = 252
    rebalance_step: int = 21
    max_weight: float = 0.35
    min_weight: float = 0.00
    target_vol_ann: float = 0.12
    transaction_cost_bps: float = 3.0
    seed: int = 42


config = RiskParityConfig()
config

## 6. Synthetic multi-asset universe

We simulate a universe with:

- equities;
- bonds;
- commodities;
- FX carry;
- trend-following;
- REITs;
- crypto proxy.

The simulation includes:

1. correlated factors;
2. changing volatility regimes;
3. stress events;
4. flight-to-quality behaviour;
5. commodity shocks;
6. high-volatility alternative asset.

This gives risk parity something realistic to handle.

In [ ]:
def simulate_risk_parity_universe(config: RiskParityConfig):
    rng = np.random.default_rng(config.seed)
    dates = pd.bdate_range("2014-01-01", periods=config.n_days)

    assets = [
        "US_EQ", "EU_EQ", "EM_EQ",
        "US_BOND", "EU_BOND",
        "GOLD", "OIL", "COPPER",
        "FX_CARRY", "TREND",
        "REIT", "BTC_PROXY"
    ]

    asset_class = {
        "US_EQ": "equity",
        "EU_EQ": "equity",
        "EM_EQ": "equity",
        "US_BOND": "bond",
        "EU_BOND": "bond",
        "GOLD": "commodity",
        "OIL": "commodity",
        "COPPER": "commodity",
        "FX_CARRY": "fx",
        "TREND": "alternative",
        "REIT": "real_asset",
        "BTC_PROXY": "crypto",
    }

    factor_names = ["market", "rates", "commodity", "carry", "trend", "crypto"]
    factor_vol_ann = np.array([0.14, 0.055, 0.13, 0.08, 0.09, 0.36])
    factor_vol_daily = factor_vol_ann / np.sqrt(config.annualisation)

    factor_corr = np.array([
        [ 1.00, -0.25,  0.25,  0.15, -0.10,  0.35],
        [-0.25,  1.00,  0.10, -0.05,  0.15, -0.20],
        [ 0.25,  0.10,  1.00,  0.10,  0.05,  0.20],
        [ 0.15, -0.05,  0.10,  1.00,  0.00,  0.15],
        [-0.10,  0.15,  0.05,  0.00,  1.00,  0.00],
        [ 0.35, -0.20,  0.20,  0.15,  0.00,  1.00],
    ])

    factor_cov = np.outer(factor_vol_daily, factor_vol_daily) * factor_corr

    loadings = pd.DataFrame(0.0, index=assets, columns=factor_names)
    loadings.loc[["US_EQ", "EU_EQ", "EM_EQ", "REIT"], "market"] = [1.00, 0.95, 1.20, 0.75]
    loadings.loc[["US_BOND", "EU_BOND"], "rates"] = [1.00, 0.90]
    loadings.loc[["GOLD", "OIL", "COPPER"], "commodity"] = [0.45, 1.10, 0.95]
    loadings.loc["FX_CARRY", "carry"] = 1.00
    loadings.loc["TREND", "trend"] = 1.00
    loadings.loc["TREND", "market"] = -0.25
    loadings.loc["BTC_PROXY", "crypto"] = 1.00
    loadings.loc["BTC_PROXY", "market"] = 0.35
    loadings.loc["GOLD", "rates"] = 0.25
    loadings.loc["REIT", "rates"] = -0.30

    idio_vol_ann = pd.Series({
        "US_EQ": 0.07,
        "EU_EQ": 0.08,
        "EM_EQ": 0.13,
        "US_BOND": 0.025,
        "EU_BOND": 0.030,
        "GOLD": 0.12,
        "OIL": 0.22,
        "COPPER": 0.18,
        "FX_CARRY": 0.07,
        "TREND": 0.08,
        "REIT": 0.10,
        "BTC_PROXY": 0.50,
    })

    annual_mean = pd.Series({
        "US_EQ": 0.07,
        "EU_EQ": 0.06,
        "EM_EQ": 0.08,
        "US_BOND": 0.025,
        "EU_BOND": 0.020,
        "GOLD": 0.035,
        "OIL": 0.045,
        "COPPER": 0.040,
        "FX_CARRY": 0.030,
        "TREND": 0.050,
        "REIT": 0.060,
        "BTC_PROXY": 0.120,
    })

    regime = np.zeros(config.n_days, dtype=int)
    transition = np.array([[0.985, 0.015], [0.055, 0.945]])

    factor_returns = np.zeros((config.n_days, len(factor_names)))
    asset_returns = np.zeros((config.n_days, len(assets)))
    stress_type = np.array(["normal"] * config.n_days, dtype=object)

    for t in range(config.n_days):
        if t > 0:
            regime[t] = rng.choice([0, 1], p=transition[regime[t - 1]])

        vol_multiplier = 1.0 if regime[t] == 0 else 2.3
        cov_t = factor_cov * vol_multiplier**2

        z = rng.standard_t(df=6, size=len(factor_names)) * np.sqrt((6 - 2) / 6)
        L = np.linalg.cholesky(cov_t + 1e-12 * np.eye(len(factor_names)))
        f = L @ z

        u = rng.uniform()
        if u < 0.008:
            stress_type[t] = "equity_crash"
            f[0] -= rng.uniform(0.035, 0.100)
            f[1] += rng.uniform(0.004, 0.025)
            f[5] -= rng.uniform(0.080, 0.220)
        elif u < 0.014:
            stress_type[t] = "inflation_shock"
            f[1] -= rng.uniform(0.010, 0.035)
            f[2] += rng.uniform(0.030, 0.090)
            f[0] -= rng.uniform(0.010, 0.050)
        elif u < 0.020:
            stress_type[t] = "commodity_crash"
            f[2] -= rng.uniform(0.040, 0.120)
            f[0] -= rng.uniform(0.005, 0.030)
        elif u < 0.025:
            stress_type[t] = "crypto_crash"
            f[5] -= rng.uniform(0.120, 0.300)
            f[0] -= rng.uniform(0.005, 0.030)

        eps = rng.standard_t(df=5, size=len(assets)) * np.sqrt((5 - 2) / 5)
        eps = eps * (idio_vol_ann.values / np.sqrt(config.annualisation))

        mu_daily = annual_mean.values / config.annualisation
        asset_returns[t] = mu_daily + loadings.to_numpy() @ f + eps
        factor_returns[t] = f

    returns_df = pd.DataFrame(asset_returns, columns=assets)
    returns_df.insert(0, "date", dates)
    returns_df["regime"] = regime
    returns_df["stress_type"] = stress_type

    factor_df = pd.DataFrame(factor_returns, columns=factor_names)
    factor_df.insert(0, "date", dates)
    factor_df["regime"] = regime
    factor_df["stress_type"] = stress_type

    metadata = pd.DataFrame({
        "asset": assets,
        "asset_class": [asset_class[a] for a in assets],
        "idio_vol_ann": [idio_vol_ann[a] for a in assets],
        "annual_mean_assumption": [annual_mean[a] for a in assets],
    })

    return returns_df, factor_df, metadata


returns_df, factor_returns, asset_metadata = simulate_risk_parity_universe(config)
asset_cols = asset_metadata["asset"].tolist()
returns = returns_df.set_index("date")[asset_cols]

returns_df.head(), asset_metadata

In [ ]:
plt.figure(figsize=(12, 6))
for asset in asset_cols:
    plt.plot(returns.index, (1 + returns[asset]).cumprod(), label=asset, alpha=0.75)
plt.title("Synthetic Asset Growth of 1")
plt.xlabel("Date")
plt.ylabel("Growth of 1")
plt.legend(ncol=3)
plt.show()

plt.figure(figsize=(12, 4))
plt.plot(returns_df["date"], returns_df["regime"], drawstyle="steps-post")
plt.title("Volatility Regime")
plt.xlabel("Date")
plt.ylabel("Regime")
plt.yticks([0, 1])
plt.show()

## 7. Return and covariance diagnostics

Before allocating by risk, inspect volatilities and correlations.

In [ ]:
return_summary = returns.agg(["mean", "std", "min", "max"]).T.rename(columns={
    "mean": "daily_mean",
    "std": "daily_vol",
    "min": "min_daily_return",
    "max": "max_daily_return",
})

return_summary["annualised_mean"] = return_summary["daily_mean"] * config.annualisation
return_summary["annualised_vol"] = return_summary["daily_vol"] * np.sqrt(config.annualisation)
return_summary["asset_class"] = return_summary.index.map(asset_metadata.set_index("asset")["asset_class"])

return_summary.sort_values("annualised_vol", ascending=False)

In [ ]:
corr = returns.corr()

plt.figure(figsize=(10, 8))
plt.imshow(corr.values, vmin=-1, vmax=1)
plt.colorbar(label="Correlation")
plt.xticks(range(len(asset_cols)), asset_cols, rotation=90)
plt.yticks(range(len(asset_cols)), asset_cols)
plt.title("Asset Return Correlation")
plt.tight_layout()
plt.show()

## 8. Covariance estimation

Risk parity depends on covariance.

We use Ledoit-Wolf shrinkage when available:

$$
\hat\Sigma_{LW}
$$

Fallback:

$$
\hat\Sigma = \delta F + (1-\delta)\hat\Sigma_{sample}
$$

where $F$ is diagonal variance target.

In [ ]:
def estimate_covariance(returns_window):
    X = returns_window.dropna().to_numpy()

    if SKLEARN_AVAILABLE:
        estimator = LedoitWolf().fit(X)
        cov = estimator.covariance_
        method = "sklearn_ledoit_wolf"
        shrinkage = float(estimator.shrinkage_)
    else:
        sample = np.cov(X, rowvar=False, ddof=1)
        target = np.diag(np.diag(sample))
        shrinkage = 0.30
        cov = shrinkage * target + (1 - shrinkage) * sample
        method = "fallback_diagonal_shrinkage"

    return cov, method, shrinkage


split_idx = int(len(returns) * config.train_fraction)
train_returns = returns.iloc[:split_idx]
test_returns = returns.iloc[split_idx:]

cov_daily, cov_method, cov_shrinkage = estimate_covariance(train_returns)
cov_ann = cov_daily * config.annualisation

pd.Series({
    "covariance_method": cov_method,
    "shrinkage": cov_shrinkage,
    "n_train_days": len(train_returns),
    "n_test_days": len(test_returns),
    "train_start": str(train_returns.index[0]),
    "train_end": str(train_returns.index[-1]),
    "test_start": str(test_returns.index[0]),
    "test_end": str(test_returns.index[-1]),
})

## 9. Risk contribution utilities

For weights $w$:

$$
MRC = \frac{\Sigma w}{\sigma_p}
$$

$$
RC = w \odot MRC
$$

$$
\%RC = \frac{RC}{\sigma_p}
$$

In [ ]:
def portfolio_volatility(weights, cov):
    w = np.asarray(weights, dtype=float)
    var = float(w.T @ cov @ w)
    return np.sqrt(max(var, 0.0))


def risk_contributions(weights, cov):
    w = np.asarray(weights, dtype=float)
    sigma = portfolio_volatility(w, cov)

    if sigma <= 0:
        mrc = np.zeros_like(w)
        rc = np.zeros_like(w)
        pct = np.zeros_like(w)
    else:
        mrc = cov @ w / sigma
        rc = w * mrc
        pct = rc / sigma

    return {
        "portfolio_vol": sigma,
        "marginal_risk_contribution": mrc,
        "component_risk_contribution": rc,
        "pct_risk_contribution": pct,
    }


def risk_contribution_table(weights, cov, assets, label):
    rc = risk_contributions(weights, cov)

    return pd.DataFrame({
        "portfolio": label,
        "asset": assets,
        "weight": weights,
        "marginal_risk_contribution": rc["marginal_risk_contribution"],
        "component_risk_contribution": rc["component_risk_contribution"],
        "pct_risk_contribution": rc["pct_risk_contribution"],
    })


equal_weights = np.ones(len(asset_cols)) / len(asset_cols)
equal_rc_table = risk_contribution_table(equal_weights, cov_daily, asset_cols, "equal_weight")
equal_rc_table.sort_values("pct_risk_contribution", ascending=False)

In [ ]:
plt.figure(figsize=(10, 6))
plot_df = equal_rc_table.sort_values("pct_risk_contribution")
plt.barh(plot_df["asset"], plot_df["pct_risk_contribution"])
plt.axvline(1 / len(asset_cols), linestyle="--", label="equal risk target")
plt.title("Risk Contributions of Equal-Weight Portfolio")
plt.xlabel("Percent risk contribution")
plt.ylabel("Asset")
plt.legend()
plt.show()

## 10. Allocation methods

We build:

1. equal weight;
2. inverse volatility;
3. global minimum variance;
4. equal risk contribution;
5. asset-class risk-budget ERC.

The ERC objective is:

$$
\min_w \sum_i (\%RC_i-b_i)^2
$$

subject to:

$$
\sum_i w_i=1
$$

$$
w_i \in [w_{min},w_{max}]
$$

In [ ]:
def normalise_weights(w, min_weight=0.0, max_weight=1.0):
    w = np.asarray(w, dtype=float)
    w = np.clip(w, min_weight, max_weight)

    if w.sum() <= 0:
        w = np.ones_like(w) / len(w)

    w = w / w.sum()

    # A second clip helps with numeric bounds; exact max can still be slightly exceeded if impossible.
    w = np.clip(w, min_weight, max_weight)
    w = w / w.sum()

    return w


def inverse_vol_weights(cov):
    vol = np.sqrt(np.diag(cov))
    inv = 1.0 / np.maximum(vol, 1e-12)
    return inv / inv.sum()


def gmv_weights(cov, min_weight, max_weight):
    n = cov.shape[0]
    x0 = np.ones(n) / n

    def obj(w):
        return float(w.T @ cov @ w)

    if SCIPY_AVAILABLE:
        res = minimize(
            obj,
            x0=x0,
            method="SLSQP",
            bounds=[(min_weight, max_weight)] * n,
            constraints=[{"type": "eq", "fun": lambda w: np.sum(w) - 1}],
            options={"maxiter": 1000, "ftol": 1e-12}
        )
        if res.success:
            return normalise_weights(res.x, min_weight, max_weight)

    # Fallback unconstrained inverse covariance then clip.
    ones = np.ones(n)
    inv_cov = np.linalg.pinv(cov)
    w = inv_cov @ ones
    w = w / np.sum(w)
    return normalise_weights(w, min_weight, max_weight)


def erc_objective(w, cov, budgets):
    pct = risk_contributions(w, cov)["pct_risk_contribution"]
    return float(np.sum((pct - budgets) ** 2))


def erc_weights(cov, budgets=None, min_weight=0.0, max_weight=0.35, seed=42):
    n = cov.shape[0]

    if budgets is None:
        budgets = np.ones(n) / n
    else:
        budgets = np.asarray(budgets, dtype=float)
        budgets = budgets / budgets.sum()

    x0 = inverse_vol_weights(cov)
    x0 = normalise_weights(x0, min_weight, max_weight)

    if SCIPY_AVAILABLE:
        res = minimize(
            lambda w: erc_objective(w, cov, budgets),
            x0=x0,
            method="SLSQP",
            bounds=[(min_weight, max_weight)] * n,
            constraints=[{"type": "eq", "fun": lambda w: np.sum(w) - 1}],
            options={"maxiter": 2000, "ftol": 1e-12}
        )
        if res.success:
            return normalise_weights(res.x, min_weight, max_weight), {
                "success": True,
                "method": "scipy_slsqp",
                "objective": float(res.fun)
            }

    # Fallback: random search around inverse vol.
    rng = np.random.default_rng(seed)
    best_w = x0
    best_obj = erc_objective(best_w, cov, budgets)

    for _ in range(25_000):
        raw = rng.dirichlet(np.ones(n))
        w = normalise_weights(raw, min_weight, max_weight)
        val = erc_objective(w, cov, budgets)
        if val < best_obj:
            best_w = w
            best_obj = val

    return best_w, {
        "success": False,
        "method": "fallback_random_search",
        "objective": float(best_obj)
    }

## 11. Asset-class risk budgets

We create class-level risk budgets, then distribute them equally across assets within each class.

Example:

| Class | Risk budget |
|---|---:|
| equity | 25% |
| bond | 20% |
| commodity | 20% |
| alternative | 15% |
| real asset | 8% |
| FX | 7% |
| crypto | 5% |

These are risk budgets, not capital weights.

In [ ]:
asset_class_map = asset_metadata.set_index("asset")["asset_class"].to_dict()

class_budgets = {
    "equity": 0.25,
    "bond": 0.20,
    "commodity": 0.20,
    "alternative": 0.15,
    "real_asset": 0.08,
    "fx": 0.07,
    "crypto": 0.05,
}

asset_budgets = pd.Series(0.0, index=asset_cols)

for cls, budget in class_budgets.items():
    assets_in_class = [a for a in asset_cols if asset_class_map[a] == cls]
    if assets_in_class:
        asset_budgets.loc[assets_in_class] = budget / len(assets_in_class)

asset_budgets = asset_budgets / asset_budgets.sum()

asset_budgets.to_frame("risk_budget")

## 12. Build static portfolios

Compute static weights on the training sample.

In [ ]:
w_equal = equal_weights
w_inv_vol = inverse_vol_weights(cov_daily)
w_gmv = gmv_weights(cov_daily, config.min_weight, config.max_weight)

w_erc, erc_info = erc_weights(
    cov_daily,
    budgets=np.ones(len(asset_cols)) / len(asset_cols),
    min_weight=config.min_weight,
    max_weight=config.max_weight,
    seed=config.seed
)

w_budget_erc, budget_erc_info = erc_weights(
    cov_daily,
    budgets=asset_budgets.to_numpy(),
    min_weight=config.min_weight,
    max_weight=config.max_weight,
    seed=config.seed + 1
)

weights_static = pd.DataFrame({
    "equal_weight": w_equal,
    "inverse_vol": w_inv_vol,
    "global_min_variance": w_gmv,
    "equal_risk_contribution": w_erc,
    "asset_class_budget_erc": w_budget_erc,
}, index=asset_cols)

weights_static

In [ ]:
plt.figure(figsize=(12, 7))
bottom = np.zeros(weights_static.shape[1])
x = np.arange(weights_static.shape[1])

for asset in weights_static.index:
    plt.bar(x, weights_static.loc[asset], bottom=bottom, label=asset)
    bottom += weights_static.loc[asset].values

plt.xticks(x, weights_static.columns, rotation=45, ha="right")
plt.title("Static Portfolio Weights")
plt.ylabel("Weight")
plt.legend(ncol=3, fontsize=8)
plt.tight_layout()
plt.show()

## 13. Risk contribution comparison

Weights can look diversified while risk is concentrated.

We compare percent risk contribution under each static portfolio.

In [ ]:
rc_tables = []

for name in weights_static.columns:
    table = risk_contribution_table(weights_static[name].to_numpy(), cov_daily, asset_cols, name)
    rc_tables.append(table)

rc_static = pd.concat(rc_tables, ignore_index=True)
rc_static.head()

In [ ]:
for name in weights_static.columns:
    sub = rc_static[rc_static["portfolio"] == name].sort_values("pct_risk_contribution")

    plt.figure(figsize=(10, 6))
    plt.barh(sub["asset"], sub["pct_risk_contribution"])
    plt.axvline(1 / len(asset_cols), linestyle="--", label="equal risk per asset")
    plt.title(f"Risk Contributions: {name}")
    plt.xlabel("Percent risk contribution")
    plt.ylabel("Asset")
    plt.legend()
    plt.show()

## 14. Portfolio-level expected risk

Using the training covariance, compare:

- expected volatility;
- concentration;
- effective number of capital bets;
- effective number of risk bets.

In [ ]:
def effective_number(x):
    x = np.asarray(x, dtype=float)
    x = np.abs(x)
    total = x.sum()
    if total <= 0:
        return np.nan
    p = x / total
    return float(1.0 / np.sum(p**2))


portfolio_risk_summary_rows = []

for name in weights_static.columns:
    w = weights_static[name].to_numpy()
    rc = risk_contributions(w, cov_daily)["pct_risk_contribution"]

    portfolio_risk_summary_rows.append({
        "portfolio": name,
        "expected_vol_ann": portfolio_volatility(w, cov_daily) * np.sqrt(config.annualisation),
        "max_weight": float(np.max(w)),
        "effective_n_capital": effective_number(w),
        "max_pct_risk_contribution": float(np.max(rc)),
        "effective_n_risk": effective_number(rc),
    })

portfolio_risk_summary = pd.DataFrame(portfolio_risk_summary_rows).sort_values("effective_n_risk", ascending=False)
portfolio_risk_summary

## 15. Static out-of-sample test

Estimate weights on train, apply to test.

Weights are static and incur a one-time entry cost.

In [ ]:
def backtest_static_weights(test_returns, weights_df, cost_bps):
    out = pd.DataFrame(index=test_returns.index)
    rows = []

    for name in weights_df.columns:
        w = weights_df[name]
        gross = test_returns @ w

        cost = pd.Series(0.0, index=test_returns.index)
        cost.iloc[0] = w.abs().sum() * cost_bps / 10000.0

        net = gross - cost
        nav = (1 + net).cumprod()
        dd = nav / nav.cummax() - 1

        out[f"{name}_return"] = net
        out[f"{name}_nav"] = nav

        rows.append({
            "portfolio": name,
            "annualised_return": float(net.mean() * config.annualisation),
            "annualised_vol": float(net.std(ddof=1) * np.sqrt(config.annualisation)),
            "sharpe_proxy": float(net.mean() / net.std(ddof=1) * np.sqrt(config.annualisation)) if net.std(ddof=1) > 0 else np.nan,
            "max_drawdown": float(dd.min()),
            "worst_day": float(net.min()),
            "total_return": float(nav.iloc[-1] - 1.0),
            "initial_cost": float(cost.iloc[0]),
        })

    return out, pd.DataFrame(rows).sort_values("sharpe_proxy", ascending=False)


static_backtest, static_performance = backtest_static_weights(test_returns, weights_static, config.transaction_cost_bps)

static_performance

In [ ]:
plt.figure(figsize=(12, 6))
for name in weights_static.columns:
    plt.plot(static_backtest.index, static_backtest[f"{name}_nav"], label=name)
plt.title("Static Out-of-Sample Portfolio Growth")
plt.xlabel("Date")
plt.ylabel("Growth of 1")
plt.legend()
plt.show()

## 16. Rolling risk parity

A practical allocation updates covariance estimates through time.

Process:

1. take trailing window;
2. estimate covariance;
3. optimise weights;
4. hold until next rebalance;
5. include turnover costs.

This creates a live-like allocation test.

In [ ]:
def rolling_allocation_weights(returns, config, method):
    dates = returns.index
    weight_rows = []
    diag_rows = []

    current = config.estimation_window

    while current < len(returns):
        window = returns.iloc[current - config.estimation_window:current]
        cov, cov_method, shrinkage = estimate_covariance(window)

        if method == "equal_weight":
            w = np.ones(len(asset_cols)) / len(asset_cols)
            info = {"method": "fixed", "objective": np.nan}
        elif method == "inverse_vol":
            w = inverse_vol_weights(cov)
            info = {"method": "closed_form", "objective": np.nan}
        elif method == "global_min_variance":
            w = gmv_weights(cov, config.min_weight, config.max_weight)
            info = {"method": "gmv", "objective": np.nan}
        elif method == "equal_risk_contribution":
            w, info = erc_weights(
                cov,
                budgets=np.ones(len(asset_cols)) / len(asset_cols),
                min_weight=config.min_weight,
                max_weight=config.max_weight,
                seed=config.seed + current
            )
        elif method == "asset_class_budget_erc":
            w, info = erc_weights(
                cov,
                budgets=asset_budgets.to_numpy(),
                min_weight=config.min_weight,
                max_weight=config.max_weight,
                seed=config.seed + current
            )
        else:
            raise ValueError("Unknown method.")

        hold_end = min(current + config.rebalance_step, len(returns))

        for date in dates[current:hold_end]:
            weight_rows.append(pd.Series(w, index=asset_cols, name=date))

        rc = risk_contributions(w, cov)["pct_risk_contribution"]

        diag_rows.append({
            "rebalance_date": dates[current],
            "method": method,
            "cov_method": cov_method,
            "shrinkage": shrinkage,
            "optimizer_method": info["method"],
            "objective": info["objective"],
            "expected_vol_ann": portfolio_volatility(w, cov) * np.sqrt(config.annualisation),
            "effective_n_capital": effective_number(w),
            "effective_n_risk": effective_number(rc),
            "max_weight": float(np.max(w)),
            "max_pct_risk_contribution": float(np.max(rc)),
        })

        current += config.rebalance_step

    weights = pd.DataFrame(weight_rows)
    weights.index.name = "date"
    weights = weights.reindex(returns.index).ffill().fillna(0.0)

    diagnostics = pd.DataFrame(diag_rows)

    return weights, diagnostics


methods = [
    "equal_weight",
    "inverse_vol",
    "global_min_variance",
    "equal_risk_contribution",
    "asset_class_budget_erc",
]

rolling_weights = {}
rolling_diagnostics = {}

for method in methods:
    w, d = rolling_allocation_weights(returns, config, method)
    rolling_weights[method] = w
    rolling_diagnostics[method] = d

rolling_diagnostics["equal_risk_contribution"].head()

## 17. Dynamic backtest with turnover costs

Weights at $t-1$ are applied to return at $t$.

Turnover cost:

$$
cost_t = c\sum_i |w_{i,t}-w_{i,t-1}|
$$

In [ ]:
def backtest_dynamic_weights(returns, weights, cost_bps):
    investable = weights.shift(1).fillna(0.0)
    gross = (investable * returns).sum(axis=1)

    turnover = weights.diff().abs().sum(axis=1).fillna(weights.abs().sum(axis=1))
    cost = turnover * cost_bps / 10000.0

    net = gross - cost
    nav = (1 + net).cumprod()

    return pd.DataFrame({
        "gross_return": gross,
        "transaction_cost": cost,
        "net_return": net,
        "nav": nav,
        "turnover": turnover,
        "gross_exposure": investable.abs().sum(axis=1),
    })


rolling_backtests = {}

for method, weights in rolling_weights.items():
    rolling_backtests[method] = backtest_dynamic_weights(returns, weights, config.transaction_cost_bps)


def summarise_backtests(backtests):
    rows = []

    for name, bt in backtests.items():
        # Start after first estimation window for fair comparison.
        sub = bt.iloc[config.estimation_window:].copy()
        r = sub["net_return"]
        nav = (1 + r).cumprod()
        dd = nav / nav.cummax() - 1

        rows.append({
            "portfolio": name,
            "annualised_return": float(r.mean() * config.annualisation),
            "annualised_vol": float(r.std(ddof=1) * np.sqrt(config.annualisation)),
            "sharpe_proxy": float(r.mean() / r.std(ddof=1) * np.sqrt(config.annualisation)) if r.std(ddof=1) > 0 else np.nan,
            "max_drawdown": float(dd.min()),
            "worst_day": float(r.min()),
            "total_return": float(nav.iloc[-1] - 1.0),
            "mean_turnover": float(sub["turnover"].mean()),
            "annualised_cost_drag": float(sub["transaction_cost"].mean() * config.annualisation),
            "mean_gross_exposure": float(sub["gross_exposure"].mean()),
        })

    return pd.DataFrame(rows).sort_values("sharpe_proxy", ascending=False)


rolling_performance = summarise_backtests(rolling_backtests)
rolling_performance

In [ ]:
plt.figure(figsize=(12, 6))
for name, bt in rolling_backtests.items():
    sub = bt.iloc[config.estimation_window:]
    nav = (1 + sub["net_return"]).cumprod()
    plt.plot(sub.index, nav, label=name)
plt.title("Rolling Allocation Backtest")
plt.xlabel("Date")
plt.ylabel("Growth of 1")
plt.legend()
plt.show()

plt.figure(figsize=(12, 5))
for name in ["inverse_vol", "equal_risk_contribution", "asset_class_budget_erc"]:
    plt.plot(rolling_backtests[name].index, rolling_backtests[name]["turnover"].rolling(21).mean(), label=name)
plt.title("Rolling 21-Day Average Turnover")
plt.xlabel("Date")
plt.ylabel("Turnover")
plt.legend()
plt.show()

## 18. Realised risk contributions

Optimised risk contributions are estimated using the training covariance.

Realised risk contributions can differ.

We estimate realised covariance over the backtest period and compute realised risk contributions using average weights.

In [ ]:
realised_cov_daily = returns.iloc[config.estimation_window:].cov().to_numpy()

realised_rc_rows = []

for name, weights in rolling_weights.items():
    avg_w = weights.iloc[config.estimation_window:].mean().to_numpy()
    rc = risk_contributions(avg_w, realised_cov_daily)["pct_risk_contribution"]

    for asset, w, rc_i in zip(asset_cols, avg_w, rc):
        realised_rc_rows.append({
            "portfolio": name,
            "asset": asset,
            "average_weight": w,
            "realised_pct_risk_contribution": rc_i,
            "asset_class": asset_class_map[asset],
        })

realised_rc = pd.DataFrame(realised_rc_rows)
realised_rc.head()

In [ ]:
for name in ["equal_weight", "inverse_vol", "equal_risk_contribution", "asset_class_budget_erc"]:
    sub = realised_rc[realised_rc["portfolio"] == name].sort_values("realised_pct_risk_contribution")

    plt.figure(figsize=(10, 6))
    plt.barh(sub["asset"], sub["realised_pct_risk_contribution"])
    plt.axvline(1 / len(asset_cols), linestyle="--", label="equal risk per asset")
    plt.title(f"Realised Risk Contributions: {name}")
    plt.xlabel("Realised percent risk contribution")
    plt.ylabel("Asset")
    plt.legend()
    plt.show()

## 19. Asset-class risk contribution

For governance, risk contributions are often monitored by asset class.

We aggregate realised risk contribution by asset class.

In [ ]:
class_rc = (
    realised_rc
    .groupby(["portfolio", "asset_class"])
    .agg(
        realised_pct_risk_contribution=("realised_pct_risk_contribution", "sum"),
        average_weight=("average_weight", "sum")
    )
    .reset_index()
)

class_rc.sort_values(["portfolio", "realised_pct_risk_contribution"], ascending=[True, False]).head(20)

In [ ]:
plot_class = class_rc.pivot(index="asset_class", columns="portfolio", values="realised_pct_risk_contribution").fillna(0.0)

plt.figure(figsize=(12, 6))
x = np.arange(len(plot_class.index))
width = 0.15

for i, portfolio in enumerate(plot_class.columns):
    plt.bar(x + (i - len(plot_class.columns)/2) * width, plot_class[portfolio], width=width, label=portfolio)

plt.xticks(x, plot_class.index, rotation=45, ha="right")
plt.title("Realised Asset-Class Risk Contributions")
plt.ylabel("Risk contribution")
plt.legend(ncol=2)
plt.tight_layout()
plt.show()

## 20. Volatility targeting risk-parity portfolios

Risk parity controls relative risk allocation, not absolute volatility.

We can scale the portfolio to a target volatility:

$$
leverage_t = \frac{\sigma_{\text{target}}} {\hat\sigma_{p,t}}
$$

Then:

$$
w^{scaled}_{i,t} = leverage_t w_{i,t}
$$

This introduces leverage and should be capped in production.

In [ ]:
def scale_weights_to_target_vol(weights, returns, config, max_leverage=3.0):
    scaled = weights.copy()

    # Estimate portfolio vol from trailing realised strategy returns.
    investable = weights.shift(1).fillna(0.0)
    raw_port_ret = (investable * returns).sum(axis=1)

    rolling_vol = raw_port_ret.rolling(config.estimation_window, min_periods=60).std() * np.sqrt(config.annualisation)
    leverage = config.target_vol_ann / rolling_vol.replace(0, np.nan)
    leverage = leverage.shift(1).fillna(1.0).clip(0.0, max_leverage)

    scaled = weights.multiply(leverage, axis=0)

    return scaled, leverage


erc_scaled_weights, erc_leverage = scale_weights_to_target_vol(
    rolling_weights["equal_risk_contribution"],
    returns,
    config,
    max_leverage=3.0
)

erc_scaled_bt = backtest_dynamic_weights(returns, erc_scaled_weights, config.transaction_cost_bps)

scaled_backtests = {
    "erc_unscaled": rolling_backtests["equal_risk_contribution"],
    "erc_target_vol_scaled": erc_scaled_bt,
}

scaled_performance = summarise_backtests(scaled_backtests)
scaled_performance

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(erc_leverage.index, erc_leverage)
plt.title("ERC Volatility Targeting Leverage")
plt.xlabel("Date")
plt.ylabel("Leverage")
plt.show()

plt.figure(figsize=(12, 6))
for name, bt in scaled_backtests.items():
    sub = bt.iloc[config.estimation_window:]
    plt.plot(sub.index, (1 + sub["net_return"]).cumprod(), label=name)
plt.title("ERC Volatility Targeting Overlay")
plt.xlabel("Date")
plt.ylabel("Growth of 1")
plt.legend()
plt.show()

## 21. Stress-regime behaviour

Risk parity can reduce concentration, but correlations rise in stress.

We compare calm and stress performance.

In [ ]:
regime_series = returns_df.set_index("date")["regime"].reindex(returns.index)
stress_type_series = returns_df.set_index("date")["stress_type"].reindex(returns.index)

stress_rows = []

for name, bt in rolling_backtests.items():
    sub = bt.iloc[config.estimation_window:].copy()
    sub["regime"] = regime_series.reindex(sub.index)
    sub["stress_type"] = stress_type_series.reindex(sub.index)

    for regime_value, g in sub.groupby("regime"):
        stress_rows.append({
            "portfolio": name,
            "regime": int(regime_value),
            "n_days": int(len(g)),
            "mean_return_ann": float(g["net_return"].mean() * config.annualisation),
            "vol_ann": float(g["net_return"].std(ddof=1) * np.sqrt(config.annualisation)),
            "worst_day": float(g["net_return"].min()),
            "mean_turnover": float(g["turnover"].mean()),
        })

stress_regime_report = pd.DataFrame(stress_rows)
stress_regime_report.head()

In [ ]:
plt.figure(figsize=(12, 6))
for name in ["equal_weight", "inverse_vol", "equal_risk_contribution", "asset_class_budget_erc"]:
    sub = stress_regime_report[stress_regime_report["portfolio"] == name]
    plt.plot(sub["regime"], sub["vol_ann"], marker="o", label=name)
plt.title("Realised Volatility by Regime")
plt.xlabel("Regime")
plt.ylabel("Annualised volatility")
plt.xticks([0, 1])
plt.legend()
plt.show()

## 22. VaR and CVaR comparison

A portfolio can have balanced volatility risk but still poor tail behaviour.

We compare historical VaR and CVaR.

In [ ]:
def historical_var_cvar(losses, alpha):
    losses = np.asarray(losses, dtype=float)
    var = np.quantile(losses, alpha)
    tail = losses[losses >= var]
    cvar = tail.mean() if len(tail) else np.nan
    return float(var), float(cvar)


tail_rows = []

for name, bt in rolling_backtests.items():
    r = bt.iloc[config.estimation_window:]["net_return"]
    losses = -r

    for alpha in [0.95, 0.99]:
        var, cvar = historical_var_cvar(losses, alpha)

        tail_rows.append({
            "portfolio": name,
            "alpha": alpha,
            "VaR": var,
            "CVaR": cvar,
        })

tail_risk_report = pd.DataFrame(tail_rows)
tail_risk_report

In [ ]:
sub = tail_risk_report[tail_risk_report["alpha"] == 0.99].sort_values("CVaR")

plt.figure(figsize=(10, 6))
plt.barh(sub["portfolio"], sub["CVaR"])
plt.title("99% Historical CVaR by Allocation Method")
plt.xlabel("CVaR loss")
plt.ylabel("Portfolio")
plt.show()

## 23. Diversification ratio

The diversification ratio is:

$$
DR = \frac{w^\top \sigma} {\sqrt{w^\top\Sigma w}}
$$

where $\sigma$ is the vector of asset volatilities.

Higher values indicate more diversification benefit.

It is not the same as risk parity, but it is a useful diagnostic.

In [ ]:
def diversification_ratio(weights, cov):
    w = np.asarray(weights, dtype=float)
    vols = np.sqrt(np.diag(cov))
    numerator = float(w @ vols)
    denominator = portfolio_volatility(w, cov)
    return numerator / denominator if denominator > 0 else np.nan


diversification_rows = []

for name in weights_static.columns:
    diversification_rows.append({
        "portfolio": name,
        "diversification_ratio_train": diversification_ratio(weights_static[name].to_numpy(), cov_daily),
    })

diversification_report = pd.DataFrame(diversification_rows).sort_values("diversification_ratio_train", ascending=False)
diversification_report

## 24. Weight stability and turnover

A risk parity optimiser can produce smoother weights than mean-variance, but rolling covariance still creates turnover.

We inspect weight volatility and turnover.

In [ ]:
weight_stability_rows = []

for name, weights in rolling_weights.items():
    weight_stability_rows.append({
        "portfolio": name,
        "mean_weight_std_across_assets": float(weights.iloc[config.estimation_window:].std().mean()),
        "max_weight_std_across_assets": float(weights.iloc[config.estimation_window:].std().max()),
        "mean_turnover": float(rolling_backtests[name].iloc[config.estimation_window:]["turnover"].mean()),
        "annualised_cost_drag": float(rolling_backtests[name].iloc[config.estimation_window:]["transaction_cost"].mean() * config.annualisation),
    })

weight_stability_report = pd.DataFrame(weight_stability_rows).sort_values("mean_turnover")
weight_stability_report

In [ ]:
plt.figure(figsize=(12, 6))
for asset in asset_cols:
    plt.plot(rolling_weights["equal_risk_contribution"].index, rolling_weights["equal_risk_contribution"][asset], label=asset, alpha=0.75)
plt.title("Rolling Equal Risk Contribution Weights")
plt.xlabel("Date")
plt.ylabel("Weight")
plt.legend(ncol=3)
plt.show()

## 25. Governance flags

A risk-parity process should flag:

1. one asset contributes too much risk;
2. one asset class dominates risk;
3. turnover is too high;
4. realised volatility exceeds target range;
5. drawdown exceeds threshold;
6. optimiser fails or hits constraints.

In [ ]:
governance_rows = []

for name in rolling_backtests.keys():
    perf = rolling_performance[rolling_performance["portfolio"] == name].iloc[0]
    rc = realised_rc[realised_rc["portfolio"] == name]

    max_asset_rc = rc["realised_pct_risk_contribution"].max()
    class_sub = class_rc[class_rc["portfolio"] == name]
    max_class_rc = class_sub["realised_pct_risk_contribution"].max()

    governance_rows.append({
        "portfolio": name,
        "annualised_vol": perf["annualised_vol"],
        "max_drawdown": perf["max_drawdown"],
        "mean_turnover": perf["mean_turnover"],
        "max_asset_risk_contribution": max_asset_rc,
        "max_class_risk_contribution": max_class_rc,
        "flag_vol_above_18pct": bool(perf["annualised_vol"] > 0.18),
        "flag_drawdown_below_minus_20pct": bool(perf["max_drawdown"] < -0.20),
        "flag_turnover_above_20pct_daily_avg": bool(perf["mean_turnover"] > 0.20),
        "flag_asset_rc_above_25pct": bool(max_asset_rc > 0.25),
        "flag_class_rc_above_50pct": bool(max_class_rc > 0.50),
    })

governance_flags = pd.DataFrame(governance_rows)
governance_flags["review_required"] = governance_flags[
    [
        "flag_vol_above_18pct",
        "flag_drawdown_below_minus_20pct",
        "flag_turnover_above_20pct_daily_avg",
        "flag_asset_rc_above_25pct",
        "flag_class_rc_above_50pct",
    ]
].any(axis=1)

governance_flags

## 26. Audit checks

Numerical checks:

1. weights sum to one for unscaled portfolios;
2. weights are finite;
3. risk contributions sum to one;
4. ERC objective is lower for ERC than equal weight;
5. no negative weights in long-only portfolios.

In [ ]:
audit_rows = []

for name, weights in weights_static.items():
    w = weights.to_numpy()
    rc_pct = risk_contributions(w, cov_daily)["pct_risk_contribution"]

    audit_rows.append({
        "check": f"{name}_weights_sum_to_one",
        "value": float(abs(w.sum() - 1)),
        "passed": bool(abs(w.sum() - 1) < 1e-8)
    })

    audit_rows.append({
        "check": f"{name}_weights_finite",
        "value": bool(np.isfinite(w).all()),
        "passed": bool(np.isfinite(w).all())
    })

    audit_rows.append({
        "check": f"{name}_risk_contributions_sum_to_one",
        "value": float(abs(rc_pct.sum() - 1)),
        "passed": bool(abs(rc_pct.sum() - 1) < 1e-8)
    })

    audit_rows.append({
        "check": f"{name}_long_only",
        "value": float(w.min()),
        "passed": bool(w.min() >= -1e-10)
    })

equal_obj = erc_objective(w_equal, cov_daily, np.ones(len(asset_cols)) / len(asset_cols))
erc_obj = erc_objective(w_erc, cov_daily, np.ones(len(asset_cols)) / len(asset_cols))

audit_rows.append({
    "check": "erc_objective_lower_than_equal_weight",
    "value": float(erc_obj - equal_obj),
    "passed": bool(erc_obj <= equal_obj + 1e-10)
})

audit_report = pd.DataFrame(audit_rows)
audit_report

## 27. Practical checklist for risk parity

Before trusting a risk-parity portfolio:

1. **Check covariance quality**  
   Risk parity is only as good as the covariance estimate.

2. **Check risk contributions**  
   Does the optimiser actually meet budgets?

3. **Check realised risk contributions**  
   Do ex-post contributions match ex-ante targets?

4. **Check asset-class concentration**  
   Equal asset risk can still mean class concentration.

5. **Check leverage**  
   Risk parity often uses leverage in low-volatility assets.

6. **Check turnover**  
   Rolling covariance creates trading.

7. **Check stress regimes**  
   Correlations rise and risk budgets can fail.

8. **Check tail risk**  
   Equal volatility contribution is not equal CVaR contribution.

9. **Check constraints**  
   Weight caps can make exact ERC impossible.

10. **Avoid expected-return blindness**  
   Risk parity does not guarantee positive expected returns.

## 28. Saving outputs

The notebook saves:

1. synthetic returns;
2. factor returns;
3. asset metadata;
4. return summary;
5. covariance matrix;
6. static weights;
7. static risk contributions;
8. portfolio risk summary;
9. static out-of-sample performance;
10. rolling weights;
11. rolling diagnostics;
12. rolling backtests;
13. realised risk contributions;
14. asset-class risk contributions;
15. scaled ERC overlay;
16. stress-regime report;
17. tail-risk report;
18. diversification report;
19. weight stability report;
20. governance flags;
21. audit report;
22. manifest.

In [ ]:
output_dir = Path("data/processed/risk_parity_and_equal_risk_contribution_v1")
output_dir.mkdir(parents=True, exist_ok=True)

returns_path = output_dir / "synthetic_returns.csv"
factor_returns_path = output_dir / "factor_returns.csv"
metadata_path = output_dir / "asset_metadata.csv"
return_summary_path = output_dir / "return_summary.csv"
cov_path = output_dir / "train_covariance_daily.csv"
weights_static_path = output_dir / "static_weights.csv"
rc_static_path = output_dir / "static_risk_contributions.csv"
portfolio_risk_summary_path = output_dir / "portfolio_risk_summary.csv"
static_performance_path = output_dir / "static_out_of_sample_performance.csv"
static_backtest_path = output_dir / "static_backtest.csv"
rolling_performance_path = output_dir / "rolling_performance.csv"
realised_rc_path = output_dir / "realised_risk_contributions.csv"
class_rc_path = output_dir / "class_risk_contributions.csv"
scaled_performance_path = output_dir / "scaled_erc_performance.csv"
erc_leverage_path = output_dir / "erc_leverage.csv"
stress_regime_path = output_dir / "stress_regime_report.csv"
tail_risk_path = output_dir / "tail_risk_report.csv"
diversification_path = output_dir / "diversification_report.csv"
weight_stability_path = output_dir / "weight_stability_report.csv"
governance_flags_path = output_dir / "governance_flags.csv"
audit_report_path = output_dir / "audit_report.csv"
manifest_path = output_dir / "manifest.json"

returns_df.to_csv(returns_path, index=False)
factor_returns.to_csv(factor_returns_path, index=False)
asset_metadata.to_csv(metadata_path, index=False)
return_summary.to_csv(return_summary_path)
pd.DataFrame(cov_daily, index=asset_cols, columns=asset_cols).to_csv(cov_path)
weights_static.to_csv(weights_static_path)
rc_static.to_csv(rc_static_path, index=False)
portfolio_risk_summary.to_csv(portfolio_risk_summary_path, index=False)
static_performance.to_csv(static_performance_path, index=False)
static_backtest.to_csv(static_backtest_path)
rolling_performance.to_csv(rolling_performance_path, index=False)
realised_rc.to_csv(realised_rc_path, index=False)
class_rc.to_csv(class_rc_path, index=False)
scaled_performance.to_csv(scaled_performance_path, index=False)
erc_leverage.to_frame("leverage").to_csv(erc_leverage_path)
stress_regime_report.to_csv(stress_regime_path, index=False)
tail_risk_report.to_csv(tail_risk_path, index=False)
diversification_report.to_csv(diversification_path, index=False)
weight_stability_report.to_csv(weight_stability_path, index=False)
governance_flags.to_csv(governance_flags_path, index=False)
audit_report.to_csv(audit_report_path, index=False)

rolling_weight_paths = {}
rolling_diag_paths = {}
rolling_bt_paths = {}

for name, weights in rolling_weights.items():
    path = output_dir / f"rolling_weights_{name}.csv"
    weights.to_csv(path)
    rolling_weight_paths[name] = str(path)

for name, diag in rolling_diagnostics.items():
    path = output_dir / f"rolling_diagnostics_{name}.csv"
    diag.to_csv(path, index=False)
    rolling_diag_paths[name] = str(path)

for name, bt in rolling_backtests.items():
    path = output_dir / f"rolling_backtest_{name}.csv"
    bt.to_csv(path)
    rolling_bt_paths[name] = str(path)

manifest = {
    "dataset_name": "risk_parity_and_equal_risk_contribution_outputs",
    "schema_version": "risk_parity_and_equal_risk_contribution_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "asset_cols": asset_cols,
    "covariance_method": cov_method,
    "covariance_shrinkage": cov_shrinkage,
    "scipy_available": SCIPY_AVAILABLE,
    "sklearn_available": SKLEARN_AVAILABLE,
    "erc_optimizer_info": erc_info,
    "budget_erc_optimizer_info": budget_erc_info,
    "rolling_performance": rolling_performance.to_dict(orient="records"),
    "governance_flags": governance_flags.to_dict(orient="records"),
    "audit_passed": bool(audit_report["passed"].all()),
    "rolling_weight_files": rolling_weight_paths,
    "rolling_diagnostic_files": rolling_diag_paths,
    "rolling_backtest_files": rolling_bt_paths,
    "notes": [
        "Equal weight, inverse volatility, GMV, equal risk contribution, and asset-class risk-budget ERC are compared.",
        "Risk contribution is calculated as w_i * (Sigma w)_i / portfolio volatility.",
        "ERC objective minimises squared distance between percent risk contributions and target budgets.",
        "Ledoit-Wolf covariance is used when sklearn is available; otherwise diagonal shrinkage fallback is used.",
        "Rolling risk parity includes turnover costs and uses weights shifted by one day before applying returns.",
        "Volatility targeting overlay scales ERC weights to a target volatility, introducing leverage.",
        "Risk parity diversifies volatility contribution, not necessarily drawdown or CVaR contribution."
    ]
}

manifest_path.write_text(json.dumps(manifest, indent=2, default=str))

returns_path, weights_static_path, rolling_performance_path, governance_flags_path, manifest_path

## 29. Limitations

### 29.1 Synthetic data

The notebook uses synthetic returns.

Real portfolios require clean total-return data, contract rolls, FX conversion, corporate actions, and investability filters.

### 29.2 Covariance estimation risk

Risk parity depends on covariance estimates.

Bad covariance estimates produce bad risk budgets.

### 29.3 Correlation instability

Risk parity can fail when correlations rise suddenly in stress.

### 29.4 No expected returns

Risk parity does not forecast returns.

Low-risk assets can be expensive or have poor expected returns.

### 29.5 Long-only constraints

The notebook uses long-only portfolios.

Some institutional risk parity portfolios use leverage and futures.

### 29.6 Simplified costs

Transaction costs are fixed bps.

Real costs vary by asset, liquidity, volatility, and market impact.

### 29.7 Volatility targeting introduces leverage

Scaling to a target volatility can increase leverage and tail risk.

### 29.8 ERC is not always feasible

Weight caps and budgets can make exact equal risk contribution impossible.

## 30. Research frontier and extensions

Important extensions include:

1. **Hierarchical risk parity**  
   Cluster assets before allocating.

2. **Equal risk contribution with full constraints**  
   Turnover, leverage, margin, and liquidity constraints.

3. **CVaR risk parity**  
   Equalise Expected Shortfall contributions instead of volatility contributions.

4. **Drawdown risk parity**  
   Allocate by drawdown contribution.

5. **Robust covariance ERC**  
   Use shrinkage, RMT denoising, or Bayesian covariance.

6. **Regime-dependent risk budgets**  
   Change budgets in stress regimes.

7. **Leverage-aware futures risk parity**  
   Add margin and collateral.

8. **Tail-risk overlays**  
   Combine ERC with put or trend-following hedges.

9. **Capacity-aware risk parity**  
   Reduce weights in illiquid markets.

10. **Chinese futures risk parity**  
   Allocate across metals, energy, agriculture, financial futures, night-session volatility, and price-limit constraints.

## 31. Suggested follow-up notebooks

This notebook naturally leads to:

1. **04_10_black_litterman_portfolio_construction**  
   Add views and priors to allocation.

2. **04_11_portfolio_constraints_margin_and_leverage**  
   Add implementation constraints.

3. **04_12_correlation_breakdown_and_regime_risk**  
   Study correlation spikes and risk budget failure.

4. **05_01_vectorized_backtest_engine**  
   Turn allocations into a reusable backtest engine.

5. **05_05_walk_forward_model_validation**  
   Formal validation of rolling allocation methods.

6. **07_01_multi_asset_cta_research_pipeline**  
   Use risk parity in a futures CTA allocation.

## 32. Summary

This notebook implemented risk parity and equal risk contribution.

It showed how to:

1. simulate multi-asset returns;
2. estimate shrinkage covariance;
3. compute marginal and component risk contributions;
4. compare equal weight and inverse volatility;
5. optimise global minimum variance;
6. optimise equal risk contribution;
7. build asset-class risk-budget ERC;
8. compare static weights and risk contributions;
9. backtest static portfolios out of sample;
10. run rolling ERC allocation with turnover costs;
11. compare realised risk contributions;
12. aggregate risk by asset class;
13. add volatility targeting to ERC;
14. analyse stress-regime behaviour;
15. compare VaR and CVaR;
16. compute diversification ratios;
17. create governance flags and audits.

The key computational takeaway:

> Equal risk contribution is an optimisation problem over covariance-driven component risk contributions.

The key financial takeaway:

> Risk parity can diversify volatility risk, but it does not eliminate correlation breakdown, tail risk, leverage risk, or poor expected returns.

## 33. Further reading

- Maillard, Roncalli, and Teiletche on equal risk contribution portfolios.
- Roncalli, *Introduction to Risk Parity and Budgeting.*
- Meucci, *Risk and Asset Allocation.*
- López de Prado on hierarchical risk parity.
- Literature on risk budgeting, ERC optimisation, covariance estimation, and volatility-managed portfolios.